# 1.05 概率与统计

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.05_probability.ipynb)


## 为什么要学

如果说线性代数是深度学习的"骨架"，概率论就是它的"灵魂"：

- **损失函数从哪来？** MSE 和交叉熵都是"最大似然估计"的结果。
- **策略是什么？** 强化学习里的策略 $\pi(a|s)$ 是一个概率分布。
- **生成模型在干嘛？** VAE、扩散、流匹配（π0 的动作头）都是在学习"动作的分布"，再从中**采样**。
- **RL 的价值函数**是"回报的期望"。

参考：[d2l 2.6 概率](https://zh.d2l.ai/chapter_preliminaries/probability.html)、d2l 附录"概率论与统计"。

---

## 1. 随机变量与常见分布

**随机变量**：取值不确定的量。用**概率分布**描述它取各个值的可能性。

| 分布 | 记号 | 取值 | 在 VLA / RL 里的例子 |
|---|---|---|---|
| 伯努利 | $\text{Bern}(p)$ | {0,1} | 夹爪开/合；任务成功/失败 |
| 类别分布 | $\text{Cat}(\mathbf{p})$ | {1..K} | 语言模型的下一个 token；离散动作 |
| 均匀分布 | $U(a,b)$ | $[a,b]$ | 随机初始化物体位置 |
| 高斯（正态） | $\mathcal{N}(\mu,\sigma^2)$ | $\mathbb{R}$ | 扩散模型的噪声；SAC 的连续动作 |
| Beta 分布 | $\text{Beta}(\alpha,\beta)$ | $[0,1]$ | **π0 训练时采样流匹配的时间 $t$** |

高斯分布的概率密度：

$$
p(x) = \frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)
$$

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import Normal, Categorical, Beta, Bernoulli

torch.manual_seed(0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].hist(Normal(0, 1).sample((10000,)).numpy(), bins=60, density=True); axes[0].set_title("Normal(0,1)")
axes[1].bar(range(4), Categorical(probs=torch.tensor([0.1, 0.2, 0.6, 0.1])).sample((10000,)).bincount() / 10000); axes[1].set_title("Categorical")
# π0 的时间采样：sample_beta(1.5, 1.0)，偏向 t 接近 1（噪声更多的那一端）
axes[2].hist(Beta(1.5, 1.0).sample((10000,)).numpy(), bins=60, density=True); axes[2].set_title("Beta(1.5, 1.0)  (pi0 time sampling)")
plt.tight_layout(); plt.show()

---

## 2. 期望、方差与蒙特卡洛估计

**期望**是"加权平均"：

$$
\mathbb{E}[X] = \sum_x x\,p(x)\quad\text{或}\quad\int x\,p(x)\,dx,\qquad
\text{Var}[X] = \mathbb{E}\big[(X-\mathbb{E}X)^2\big] = \mathbb{E}[X^2] - (\mathbb{E}X)^2
$$

**大数定律**：独立采样 $N$ 次取平均，会收敛到期望，误差约为 $\sigma/\sqrt{N}$。

$$
\mathbb{E}[f(X)] \approx \frac{1}{N}\sum_{i=1}^N f(x_i),\quad x_i\sim p
$$

这就是**蒙特卡洛估计**。强化学习里"不知道环境模型，就多玩几局取平均"（第 2 章的 MC 方法）就是它。

In [ ]:
# 估计 E[X^2]，X ~ N(1, 2^2)。真实值 = Var + mean^2 = 4 + 1 = 5
for N in [10, 100, 1000, 100000]:
    xs = Normal(1.0, 2.0).sample((N,))
    print(f"N={N:6d}  估计值 {(xs ** 2).mean():.4f}")

### 协方差

两个随机变量的协方差 $\text{Cov}(X,Y) = \mathbb{E}[(X-\mathbb{E}X)(Y-\mathbb{E}Y)]$，衡量它们是否"一起变大变小"。多维高斯 $\mathcal{N}(\boldsymbol\mu, \Sigma)$ 的协方差矩阵 $\Sigma$ 描述各维度的相关性。

---

## 3. 条件概率与贝叶斯公式

$$
p(A|B) = \frac{p(A,B)}{p(B)},\qquad p(A|B) = \frac{p(B|A)\,p(A)}{p(B)}
$$

- 策略 $\pi(a|s)$ 是**条件分布**："在状态 $s$ 下选动作 $a$ 的概率"。
- VLA 学的是 $p(\text{动作块}\mid\text{图像},\text{指令},\text{本体状态})$。
- 环境转移 $p(s'|s,a)$ 也是条件分布。

---

## 4. 最大似然估计（MLE）：损失函数的来源

给定数据 $\{x_i\}$，找到参数 $\theta$ 让"数据出现的概率"最大：

$$
\theta^* = \arg\max_\theta \prod_i p_\theta(x_i) = \arg\min_\theta \underbrace{-\sum_i \log p_\theta(x_i)}_{\text{负对数似然 NLL}}
$$

**两个最重要的推论**：

**(1) 高斯似然 ⇒ MSE。** 若假设 $y \sim \mathcal{N}(f_\theta(x), \sigma^2)$，$\sigma$ 固定，则

$$
-\log p(y|x) = \frac{(y - f_\theta(x))^2}{2\sigma^2} + \text{常数}
$$

所以**最小化 MSE = 在高斯假设下做最大似然**。这也说明了 MSE 的局限：它假设分布只有**一个峰**。如果专家有时往左、有时往右（双峰），MSE 会学到"中间"。第 3 章会亲眼看到这个问题。

**(2) 类别似然 ⇒ 交叉熵。** 若 $y\sim\text{Cat}(\mathbf{p}_\theta(x))$，则 $-\log p(y|x) = -\log p_{\theta,y}(x)$，这正是分类用的交叉熵损失。

---

## 5. 熵、交叉熵、KL 散度

$$
H(p) = -\sum_x p(x)\log p(x),\qquad
H(p, q) = -\sum_x p(x)\log q(x),\qquad
D_{KL}(p\|q) = \sum_x p(x)\log\frac{p(x)}{q(x)} = H(p,q) - H(p)
$$

- **熵** $H(p)$：分布的"不确定性"。均匀分布熵最大。SAC 在奖励里加熵，鼓励探索（2.12 课）。
- **KL 散度**：两个分布的"距离"（不对称、非负，$p=q$ 时为 0）。VAE、PPO、RLHF 里到处都是。

两个对角高斯之间的 KL 有闭式解（VAE 和 ACT 的 KL 损失就用这个）：

$$
D_{KL}\big(\mathcal{N}(\mu,\sigma^2)\,\|\,\mathcal{N}(0,1)\big) = \frac{1}{2}\left(\mu^2 + \sigma^2 - 1 - \log\sigma^2\right)
$$

In [ ]:
p = torch.tensor([0.7, 0.2, 0.1]); q = torch.tensor([0.4, 0.4, 0.2]); u = torch.ones(3) / 3
H = lambda p: -(p * p.log()).sum()
KL = lambda p, q: (p * (p / q).log()).sum()
print(f"H(p)={H(p):.3f}  H(均匀)={H(u):.3f} (= log 3 = {np.log(3):.3f})")
print(f"KL(p||q)={KL(p, q):.3f}  KL(q||p)={KL(q, p):.3f}  (不对称)  KL(p||p)={KL(p, p):.3f}")

---

## 6. 重参数化技巧（Reparameterization Trick）

想对 $\mathbb{E}_{z\sim\mathcal{N}(\mu,\sigma^2)}[f(z)]$ 关于 $\mu,\sigma$ 求梯度，但"采样"这个操作不可导。技巧是把随机性"挪出去"：

$$
z = \mu + \sigma\cdot\epsilon,\qquad \epsilon\sim\mathcal{N}(0,1)
$$

现在 $z$ 是 $\mu,\sigma$ 的可导函数，随机性只在 $\epsilon$ 里。**VAE、ACT 的 CVAE、SAC 的策略更新**全都用它。PyTorch 中 `dist.rsample()`（r = reparameterized）就是这个，而 `dist.sample()` 不可导。

In [ ]:
mu = torch.tensor(0.5, requires_grad=True); log_sigma = torch.tensor(0.0, requires_grad=True)
eps = torch.randn(10000)
z = mu + log_sigma.exp() * eps          # 重参数化采样
loss = (z ** 2).mean()                   # 估计 E[z^2] = mu^2 + sigma^2
loss.backward()
print("dE/dmu ≈", mu.grad.item(), "(理论 2*mu = 1.0)")
print("dE/dlog_sigma ≈", log_sigma.grad.item(), "(理论 2*sigma^2 = 2.0)")

---

## 练习

### 练习 1：为什么注意力要除以 √d？

设 $\mathbf{q},\mathbf{k}\in\mathbb{R}^d$，每个分量独立服从 $\mathcal{N}(0,1)$。用蒙特卡洛估计 $\mathbf{q}^\top\mathbf{k}$ 的方差，并验证它约等于 $d$。所以除以 $\sqrt d$ 后方差变回 1，softmax 就不会因为数值太大而变成"one-hot"（梯度消失）。

In [ ]:
def dot_product_variance(d, n_samples=20000):
    """返回 q·k 的样本方差（q, k ~ N(0, I_d)）"""
    # TODO: 采样 q, k 形状 (n_samples, d)，逐行点积，求方差
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def dot_product_variance(d, n_samples=20000):
    q = torch.randn(n_samples, d); k = torch.randn(n_samples, d)
    dots = (q * k).sum(dim=-1)        # (n_samples,)
    return dots.var().item()

In [ ]:
# ✅ 自测
for d in [16, 64, 256]:
    v = dot_product_variance(d)
    print(f"d={d:3d}  Var(q·k)≈{v:7.1f}  Var(q·k/√d)≈{v / d:.3f}")
    assert abs(v / d - 1) < 0.1
print("通过 ✅ 方差 ≈ d，除以 √d 后 ≈ 1")

> 推导：$\text{Var}(\sum_i q_ik_i) = \sum_i \text{Var}(q_ik_i) = \sum_i \mathbb{E}[q_i^2]\mathbb{E}[k_i^2] = d$。

### 练习 2：手写高斯对数概率

实现对角高斯的对数概率密度：$\mathbf{x}\in\mathbb{R}^D$，均值 $\boldsymbol\mu$，标准差 $\boldsymbol\sigma$，各维独立：

$$
\log p(\mathbf{x}) = \sum_{i=1}^D\left[-\frac{(x_i-\mu_i)^2}{2\sigma_i^2} - \log\sigma_i - \frac{1}{2}\log 2\pi\right]
$$

RL 里计算策略的 $\log\pi(a|s)$（PPO、SAC）就是这个公式。

In [ ]:
def gaussian_log_prob(x, mu, sigma):
    """x, mu, sigma: (..., D) -> (...,)  在最后一维上求和"""
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def gaussian_log_prob(x, mu, sigma):
    """x, mu, sigma: (..., D) -> (...,)  在最后一维上求和"""
    lp = -((x - mu) ** 2) / (2 * sigma ** 2) - torch.log(sigma) - 0.5 * np.log(2 * np.pi)
    return lp.sum(dim=-1)

In [ ]:
# ✅ 自测
x = torch.randn(8, 3); mu = torch.randn(8, 3); sigma = torch.rand(8, 3) + 0.1
ref = Normal(mu, sigma).log_prob(x).sum(-1)
assert torch.allclose(gaussian_log_prob(x, mu, sigma), ref, atol=1e-5)
print("通过 ✅")

### 练习 3：高斯 KL 的闭式解 vs 蒙特卡洛

实现 $D_{KL}(\mathcal{N}(\mu,\sigma^2)\|\mathcal{N}(0,1))$ 的闭式解（对最后一维求和），并用蒙特卡洛（$\mathbb{E}_{z\sim q}[\log q(z) - \log p(z)]$）验证。

In [ ]:
def kl_to_standard_normal(mu, logvar):
    """mu, logvar: (..., D)，其中 logvar = log σ²。返回 (...,)"""
    # TODO: 0.5 * sum(mu^2 + exp(logvar) - 1 - logvar)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def kl_to_standard_normal(mu, logvar):
    """mu, logvar: (..., D)，其中 logvar = log σ²。返回 (...,)"""
    return 0.5 * (mu ** 2 + logvar.exp() - 1 - logvar).sum(dim=-1)

In [ ]:
# ✅ 自测
mu = torch.tensor([0.5, -1.0]); logvar = torch.tensor([0.2, -0.5])
closed = kl_to_standard_normal(mu, logvar)
q = Normal(mu, (0.5 * logvar).exp()); p = Normal(torch.zeros(2), torch.ones(2))
z = q.sample((200000,))
mc = (q.log_prob(z) - p.log_prob(z)).sum(-1).mean()
print(f"闭式解 {closed:.4f}   蒙特卡洛 {mc:.4f}")
assert abs(closed - mc) < 0.02
print("通过 ✅")

---

## 小结

| 概念 | 公式 | 在哪用 |
|---|---|---|
| 期望/蒙特卡洛 | $\mathbb{E}f \approx \frac1N\sum f(x_i)$ | RL 的回报估计 |
| 最大似然 | $\min -\sum\log p_\theta(x_i)$ | MSE（高斯）、交叉熵（类别） |
| 熵 | $-\sum p\log p$ | SAC 的探索奖励 |
| KL 散度 | $\sum p\log\frac pq$ | VAE/ACT、PPO、RLHF |
| 重参数化 | $z=\mu+\sigma\epsilon$ | VAE、SAC |

**思考题**：π0 训练时用 $\text{Beta}(1.5, 1)$ 采样时间 $t$，让 $t$ 更多地落在接近 1（纯噪声）的一端。为什么"更多地练习从纯噪声开始去噪"可能对动作生成有好处？（第 6 章揭晓。）